# Document OCR with MiniCPM-OCR and OpenVINO

MiniCPM-OCR is a document-understanding model from the [MiniCPM-V](https://huggingface.co/papers/2509.18154) family developed by [OpenBMB](https://github.com/OpenBMB). It keeps the [MiniCPM-V 4.6](../minicpm-v-4.6/minicpm-v-4.6.ipynb) architecture, a SigLIP vision encoder with a window-attention merger in front of a Qwen3.5 language model, and specializes it for reading text-dense images such as scanned pages, screenshots, receipts and tables. See the [Transformers documentation](https://huggingface.co/docs/transformers/main/model_doc/minicpm_ocr) for details.

In this tutorial we export [`openbmb/MiniCPM-OCR-1.0`](https://huggingface.co/openbmb/MiniCPM-OCR-1.0) to OpenVINO IR with [Optimum Intel](https://github.com/huggingface/optimum-intel/pull/2081), compress its weights with [NNCF](https://github.com/openvinotoolkit/nncf), and run inference with `OVModelForVisualCausalLM`: we transcribe a receipt, convert a form to Markdown, read Chinese text from an image crop and answer a question about a document. Finally, we launch an interactive Gradio demo.

#### Table of contents:

- [Prerequisites](#Prerequisites)
- [Convert and compress the model](#Convert-and-compress-the-model)
- [Load the model with OpenVINO](#Load-the-model-with-OpenVINO)
- [Run OCR](#Run-OCR)
    - [Transcribe a receipt](#Transcribe-a-receipt)
    - [Convert a page to Markdown](#Convert-a-page-to-Markdown)
    - [Read multilingual text](#Read-multilingual-text)
    - [Ask a question about a document](#Ask-a-question-about-a-document)
- [Measure accuracy and performance](#Measure-accuracy-and-performance)
- [Interactive demo](#Interactive-demo)

⚠️ **EXPERIMENTAL NOTEBOOK**

This notebook demonstrates a model that has not been fully validated with OpenVINO and uses development builds of Optimum Intel, Transformers and OpenVINO. It may be fully supported and validated in the future.

<img referrerpolicy="no-referrer-when-downgrade" src="https://static.scarf.sh/a.png?x-pxid=5b5a4db0-7875-4bfb-bdbd-01698b5b1a77&file=notebooks/minicpm-ocr/minicpm-ocr.ipynb" />

### Installation Instructions

This is a self-contained example that relies solely on its own code.

We recommend running the notebook in a virtual environment. You only need a Jupyter server to start.
For details, please refer to [Installation Guide](https://github.com/openvinotoolkit/openvino_notebooks/blob/latest/README.md#-installation-guide).

## Prerequisites
[back to top ⬆️](#Table-of-contents:)

MiniCPM-OCR export is provided by the [Optimum Intel pull request](https://github.com/huggingface/optimum-intel/pull/2081), and the model itself is available in Transformers starting from the main branch. The pull request declares an older Transformers requirement, so Transformers is installed after Optimum Intel. OpenVINO and OpenVINO Tokenizers are installed from nightly builds. Restart the kernel after installation if any of these packages were already imported.

In [ ]:
%pip uninstall -q -y optimum optimum-intel optimum-onnx
%pip install -q "git+https://github.com/rkazants/optimum-intel.git@minicpm_ocr" --extra-index-url https://download.pytorch.org/whl/cpu
%pip install -q "git+https://github.com/huggingface/transformers.git" --extra-index-url https://download.pytorch.org/whl/cpu
%pip install -q "torch>=2.4" torchvision nncf datasets Pillow requests ipywidgets huggingface_hub "gradio>=5,<7" --extra-index-url https://download.pytorch.org/whl/cpu
%pip install -q --pre -U openvino openvino-tokenizers --extra-index-url https://storage.openvinotoolkit.org/simple/wheels/nightly

In [ ]:
from pathlib import Path

import requests

for name in ("cmd_helper.py", "notebook_utils.py"):
    if not Path(name).exists():
        response = requests.get(
            f"https://raw.githubusercontent.com/openvinotoolkit/openvino_notebooks/latest/utils/{name}",
            timeout=30,
        )
        response.raise_for_status()
        Path(name).write_text(response.text, encoding="utf-8")

# Read more about telemetry collection at https://github.com/openvinotoolkit/openvino_notebooks?tab=readme-ov-file#-telemetry
from notebook_utils import collect_telemetry

collect_telemetry("minicpm-ocr.ipynb")

## Convert and compress the model
[back to top ⬆️](#Table-of-contents:)

MiniCPM-OCR is a PyTorch model. We convert it to OpenVINO Intermediate Representation (IR) with the `optimum-cli` tool of [Optimum Intel](https://huggingface.co/docs/optimum/intel/index):

```bash
optimum-cli export openvino --model <model_id_or_path> --task image-text-to-text --weight-format <format> <output_dir>
```

The model is exported as three OpenVINO models: the vision encoder, the text embeddings and the Qwen3.5 language model. Weight compression with [NNCF](https://github.com/openvinotoolkit/nncf) reduces their memory footprint. OCR needs character-exact output, so the weight format affects quality more than for a conversational model:

* `fp16` stays closest to the original model and needs the most memory.
* `int8` (default) is a balance between size and quality.
* `int4` applies the data-aware recipe that Optimum Intel defines for this checkpoint (AWQ and scale estimation calibrated on TextVQA samples). It is the smallest and fastest option, takes longer to export and downloads the calibration dataset.

Accuracy and speed depend on the weight format, the device and the software versions; the [Measure accuracy and performance](#Measure-accuracy-and-performance) section measures them for your setup.

`Model:` accepts a Hugging Face model ID or a local checkpoint directory.

In [ ]:
import ipywidgets as widgets

model_source = widgets.Text(value="openbmb/MiniCPM-OCR-1.0", description="Model:")
weight_format = widgets.Dropdown(options=["int8", "int4", "fp16"], value="int8", description="Weights:")
display(model_source, weight_format)

In [ ]:
import os
import sys

from cmd_helper import optimum_cli

model_id = model_source.value.strip()
model_dir = Path(f"{Path(model_id).name}-ov") / weight_format.value

if not (model_dir / "openvino_language_model.xml").exists():
    os.environ["PATH"] = str(Path(sys.executable).parent) + os.pathsep + os.environ["PATH"]
    optimum_cli(model_id, model_dir, additional_args={"task": "image-text-to-text", "weight-format": weight_format.value})
print(f"OpenVINO model: {model_dir}")

## Load the model with OpenVINO
[back to top ⬆️](#Table-of-contents:)

`OVModelForVisualCausalLM` loads the three OpenVINO models and provides the familiar Transformers `generate` API, while the Transformers processor prepares image slices and the chat prompt.

GPU runs in f16 inference precision by default, which can change single characters of a transcription. Select **Full precision on GPU** to request f32 inference precision at the cost of speed.

In [ ]:
from notebook_utils import device_widget

device = device_widget(default="AUTO", exclude=["NPU"])
gpu_full_precision = widgets.Checkbox(value=False, description="Full precision on GPU")
display(device, gpu_full_precision)

In [ ]:
from optimum.intel import OVModelForVisualCausalLM
from transformers import AutoProcessor

ov_config = {"INFERENCE_PRECISION_HINT": "f32"} if gpu_full_precision.value and "GPU" in device.value else None

processor = AutoProcessor.from_pretrained(model_dir)
ov_model = OVModelForVisualCausalLM.from_pretrained(model_dir, device=device.value, ov_config=ov_config)
print(f"Model loaded on {device.value}, visual downsampling: {ov_model.config.downsample_mode}")

## Run OCR
[back to top ⬆️](#Table-of-contents:)

The processor applies the model chat template, slices the image and inserts the visual placeholders. A few details are specific to MiniCPM-OCR:

* **Visual downsampling.** The `4x` mode (default) keeps 4× more visual tokens than `16x`, which is what document and screenshot transcription needs. The OpenVINO vision encoder is exported for the mode from the model config, so the processor must use the same mode; the `ocr` function below passes it explicitly.
* **Thinking.** The chat template opens a thinking block unless `enable_thinking=False` is passed. Plain transcription does not need it, so it is disabled by default.
* **Greedy decoding** (`do_sample=False`) keeps transcriptions deterministic.

In [ ]:
from io import BytesIO

from PIL import Image
from transformers import TextStreamer


def load_image(url, path):
    path = Path(path)
    if not path.exists():
        response = requests.get(url, timeout=30)
        response.raise_for_status()
        path.write_bytes(response.content)
    return Image.open(path).convert("RGB")


def ocr(image, prompt, *, enable_thinking=False, max_new_tokens=512):
    messages = [{"role": "user", "content": [{"type": "image", "image": image}, {"type": "text", "text": prompt}]}]
    inputs = processor.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=True,
        return_dict=True,
        return_tensors="pt",
        enable_thinking=enable_thinking,
        processor_kwargs={"downsample_mode": ov_model.config.downsample_mode},
    )
    streamer = TextStreamer(processor.tokenizer, skip_prompt=True, skip_special_tokens=True)
    output_ids = ov_model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False, streamer=streamer)
    return processor.decode(output_ids[0, inputs["input_ids"].shape[1] :], skip_special_tokens=True).strip()

### Transcribe a receipt
[back to top ⬆️](#Table-of-contents:)

In [ ]:
receipt = load_image(
    "https://huggingface.co/datasets/huggingface/documentation-images/resolve/main/transformers/model_doc/ocr_receipt.jpeg",
    "ocr_receipt.jpeg",
)
display(receipt.resize((receipt.width * 512 // receipt.height, 512)))
receipt_text = ocr(receipt, "Transcribe all the text in this document.")

### Convert a page to Markdown
[back to top ⬆️](#Table-of-contents:)

The model can also restore the document structure, such as headings, fields and lists, in Markdown.

In [ ]:
form = load_image(
    "https://huggingface.co/datasets/huggingface/documentation-images/resolve/main/transformers/model_doc/ocr_suggestion_form.jpg",
    "ocr_suggestion_form.jpg",
)
display(form.resize((form.width * 640 // form.height, 640)))
form_markdown = ocr(form, "Convert this page to markdown.", max_new_tokens=1024)

### Read multilingual text
[back to top ⬆️](#Table-of-contents:)

Text is often read from regions cut out by a layout analysis step. Here we crop a Chinese caption from the example image of the [PaddleOCR Space](https://huggingface.co/spaces/PaddlePaddle/PaddleOCR).

In [ ]:
comic = load_image("https://huggingface.co/spaces/PaddlePaddle/PaddleOCR/resolve/main/cn_example.jpg", "cn_example.jpg")
caption = comic.crop((700, 525, 1135, 820))
display(caption)
caption_text = ocr(caption, "Transcribe all the text in this image.", max_new_tokens=128)

### Ask a question about a document
[back to top ⬆️](#Table-of-contents:)

Besides transcription, the model answers questions about the document content. Pass `enable_thinking=True` to let the model reason before answering; increase `max_new_tokens` accordingly.

In [ ]:
answer = ocr(receipt, "What is the document number and the total amount on this receipt?", max_new_tokens=128)

## Measure accuracy and performance
[back to top ⬆️](#Table-of-contents:)

Weight compression and reduced inference precision can change characters of a transcription. Here we run the examples above with the OpenVINO model and with the original PyTorch model and compare:

* **Similarity**: character-level similarity of the OpenVINO and PyTorch transcriptions (`difflib.SequenceMatcher` ratio, 1.0 means identical text).
* **Identical answers**: the number of examples with exactly the same output.
* **Time to first token**: image encoding and prompt processing of the OpenVINO model.
* **Throughput**: generated tokens per second after the first token.

The PyTorch model is loaded in FP32 on CPU, which needs about 5 GB of RAM for MiniCPM-OCR-1.0. Clear the checkbox to measure only the performance of the OpenVINO model. Answers are limited to `max_new_tokens=256` to keep the measurement short.

In [ ]:
compare_with_pytorch = widgets.Checkbox(value=True, description="Compare with the original PyTorch model")
compare_with_pytorch

In [ ]:
import gc
import time
from difflib import SequenceMatcher

import torch
from transformers import AutoModelForImageTextToText
from transformers.generation.streamers import BaseStreamer


class TimingStreamer(BaseStreamer):
    # `generate` puts the prompt first, then every generated token.
    def __init__(self):
        self.times = []

    def put(self, value):
        self.times.append(time.perf_counter())

    def end(self):
        pass


def transcribe(model, image, prompt, max_new_tokens=256, streamer=None):
    messages = [{"role": "user", "content": [{"type": "image", "image": image}, {"type": "text", "text": prompt}]}]
    inputs = processor.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=True,
        return_dict=True,
        return_tensors="pt",
        enable_thinking=False,
        processor_kwargs={"downsample_mode": ov_model.config.downsample_mode},
    )
    output_ids = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False, streamer=streamer)
    return processor.decode(output_ids[0, inputs["input_ids"].shape[1] :], skip_special_tokens=True).strip()


samples = {
    "receipt": (receipt, "Transcribe all the text in this document."),
    "form": (form, "Convert this page to markdown."),
    "caption": (caption, "Transcribe all the text in this image."),
    "question": (receipt, "What is the document number and the total amount on this receipt?"),
}

transcribe(ov_model, caption, "Transcribe all the text in this image.", max_new_tokens=8)  # warm-up

ov_results = {}
for name, (image, prompt) in samples.items():
    streamer = TimingStreamer()
    text = transcribe(ov_model, image, prompt, streamer=streamer)
    first_token, last_token = streamer.times[1], streamer.times[-1]
    new_tokens = len(streamer.times) - 2
    ov_results[name] = {
        "text": text,
        "ttft": first_token - streamer.times[0],
        "throughput": new_tokens / (last_token - first_token) if new_tokens else float("nan"),
    }

if compare_with_pytorch.value:
    pt_model = AutoModelForImageTextToText.from_pretrained(model_id, dtype=torch.float32).eval()
    with torch.no_grad():
        for name, (image, prompt) in samples.items():
            ov_results[name]["similarity"] = SequenceMatcher(None, transcribe(pt_model, image, prompt), ov_results[name]["text"]).ratio()
    del pt_model
    gc.collect()

print(f"{'example':<10} {'similarity':>10} {'TTFT, s':>8} {'tokens/s':>9}")
for name, result in ov_results.items():
    similarity = f"{result['similarity']:.4f}" if "similarity" in result else "-"
    print(f"{name:<10} {similarity:>10} {result['ttft']:>8.2f} {result['throughput']:>9.1f}")
if compare_with_pytorch.value:
    similarities = [result["similarity"] for result in ov_results.values()]
    print(f"\nMean similarity: {sum(similarities) / len(similarities):.4f}")
    print(f"Identical answers: {sum(similarity == 1.0 for similarity in similarities)}/{len(similarities)}")
print(f"Device: {device.value}, weights: {model_dir}")

## Interactive demo
[back to top ⬆️](#Table-of-contents:)

Upload an image or select an example, choose a prompt and run OCR. The answer is streamed as it is generated.

In [ ]:
if not Path("gradio_helper.py").exists():
    response = requests.get(
        "https://raw.githubusercontent.com/openvinotoolkit/openvino_notebooks/latest/notebooks/minicpm-ocr/gradio_helper.py",
        timeout=30,
    )
    response.raise_for_status()
    Path("gradio_helper.py").write_text(response.text, encoding="utf-8")

from gradio_helper import make_demo

demo = make_demo(ov_model, processor, examples=["ocr_receipt.jpeg", "ocr_suggestion_form.jpg", "cn_example.jpg"])

try:
    demo.launch(debug=True)
except Exception:
    demo.launch(debug=True, share=True)
# If you are launching remotely, specify server_name and server_port
# EXAMPLE: `demo.launch(server_name='your server name', server_port='server port in int')`
# To learn more please refer to the Gradio docs: https://gradio.app/docs/